In [2]:
import json 

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
# the housing units for each community district
housing = pd.read_csv("cd_housing_units.csv")

# the population for each ZIP area. dtype={"zip": str} keeps the ZIP codes as text, to match the shapes
zips = pd.read_csv("zip_population.csv", dtype={"zip": str})

# the shapes: community districts, then ZIP areas
with open("nyc_districts.geojson", "r", encoding="utf-8") as file:
    district_shapes = json.load(file)

with open("zip_areas.geojson", "r", encoding="utf-8") as file:
    zip_shapes = json.load(file)

print(len(housing), "community district rows,", len(zips), "ZIP areas")

In [ ]:
# keep only the 59 community districts: the rest are parks and airports, which have almost no housing
districts = housing[housing["is_district"]]

districts.head()

In [ ]:
zips.head()

In [ ]:
# the text in the hover box for each district. <br> starts a new line.
# zip() goes through several columns together, one row at a time: first the first label with the first number
# of units, then the second of each, and so on
district_hover = []

for label, units in zip(districts["label"], districts["housing_units"]):
    district_hover.append(f"<b>{label}</b><br>Housing units: {units}")

housing_layer = go.Choroplethmap(
    geojson=district_shapes,
    locations=districts["commntydst"],       # the column that matches...
    featureidkey="properties.commntydst",    # ...this property of each shape
    z=districts["housing_units"],            # the number that decides each district's color
    colorscale="Blues",
    marker_opacity=0.75,
    marker_line_color="white",               # white outlines between the districts
    marker_line_width=1,
    colorbar={"title": "Housing units<br>(2020 census)", "x": 1.0, "len": 0.5},
    text=district_hover,
    hoverinfo="text",                        # show only our text, not the raw numbers
    showlegend=True,
    name="Housing units (community districts)",
)

In [ ]:
fig = go.Figure(housing_layer)

fig.update_layout(
    map_style="carto-positron",
    map_center={"lat": 40.71, "lon": -73.95},     # the middle of New York City
    map_zoom=9.5,
    height=650,
    margin={"l": 0, "r": 0, "t": 0, "b": 0},
)

fig.show()